In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Dashboard imports
import dash_leaflet as dl
from dash import dcc, html
from dash import dash_table
from dash.dependencies import Input, Output
import plotly.express as px

# Data imports
import pandas as pd
import base64
import os

# Import CRUD module
from animal_shelter import AnimalShelter

# Configure JupyterDash
JupyterDash.infer_jupyter_proxy_config()


###########################
# Data Manipulation / Model
###########################

username = "aacuser"
password = "fatima"

shelter = AnimalShelter(username, password)


def clean_dataframe(records):
    """Convert MongoDB records into a clean dataframe for Dash."""
    df = pd.DataFrame.from_records(records)

    if "_id" in df.columns:
        df.drop(columns=["_id"], inplace=True)

    return df


def get_query(filter_type):
    """Return MongoDB query based on rescue type filter."""

    if filter_type == "water":
        return {
            "animal_type": "Dog",
            "breed": {
                "$regex": "Labrador Retriever|Chesapeake Bay Retriever|Newfoundland",
                "$options": "i"
            },
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {
                "$gte": 26,
                "$lte": 156
            }
        }

    elif filter_type == "mountain":
        return {
            "animal_type": "Dog",
            "breed": {
                "$regex": "German Shepherd|Alaskan Malamute|Old English Sheepdog|Siberian Husky|Rottweiler",
                "$options": "i"
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {
                "$gte": 26,
                "$lte": 156
            }
        }

    elif filter_type == "disaster":
        return {
            "animal_type": "Dog",
            "breed": {
                "$regex": "Doberman Pinscher|German Shepherd|Golden Retriever|Bloodhound|Rottweiler",
                "$options": "i"
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {
                "$gte": 20,
                "$lte": 300
            }
        }

    else:
        return {}


df = clean_dataframe(shelter.read({}))


#########################
# Logo Setup
#########################

image_filename = "Grazioso Salvare Logo.png"

encoded_image = ""

if os.path.exists(image_filename):
    with open(image_filename, "rb") as image_file:
        encoded_image = base64.b64encode(image_file.read()).decode()


#########################
# Dashboard Layout / View
#########################

app = JupyterDash("GraziosoSalvareDashboard")

app.layout = html.Div(
    style={
        "backgroundColor": "#f7f2f4",
        "fontFamily": "Arial, sans-serif",
        "padding": "20px"
    },
    children=[

        html.Div(
            style={
                "backgroundColor": "white",
                "borderRadius": "18px",
                "padding": "25px",
                "boxShadow": "0 4px 12px rgba(0,0,0,0.15)",
                "borderTop": "10px solid #b1123f"
            },
            children=[

                html.Center([
                    html.A(
                        html.Img(
                            src="data:image/png;base64,{}".format(encoded_image),
                            style={
                                "height": "180px",
                                "width": "180px"
                            }
                        ),
                        href="https://www.snhu.edu",
                        target="_blank"
                    )
                ]),

                html.Center(
                    html.H1(
                        "SNHU CS-340 Dashboard",
                        style={
                            "color": "#7a0026",
                            "fontWeight": "bold",
                            "marginBottom": "5px"
                        }
                    )
                ),

                html.Center(
                    html.H3(
                        "Created by Fatima Mubasher",
                        style={
                            "color": "#333333",
                            "marginTop": "0px"
                        }
                    )
                ),

                html.Hr(),

                html.Div(
                    style={
                        "backgroundColor": "#fce7ef",
                        "border": "2px solid #b1123f",
                        "borderRadius": "12px",
                        "padding": "15px",
                        "marginBottom": "20px"
                    },
                    children=[
                        html.H3(
                            "Interactive Filter Options",
                            style={
                                "color": "#7a0026",
                                "marginTop": "0px"
                            }
                        ),

                        dcc.RadioItems(
                            id="filter-type",
                            options=[
                                {"label": "Water Rescue", "value": "water"},
                                {"label": "Mountain or Wilderness Rescue", "value": "mountain"},
                                {"label": "Disaster or Individual Tracking", "value": "disaster"},
                                {"label": "Reset", "value": "reset"}
                            ],
                            value="reset",
                            labelStyle={
                                "display": "inline-block",
                                "marginRight": "25px",
                                "fontWeight": "bold",
                                "color": "#333333"
                            }
                        ),
                    ]
                ),

                dash_table.DataTable(
                    id="datatable-id",

                    columns=[
                        {"name": i, "id": i, "deletable": False, "selectable": True}
                        for i in df.columns
                    ],

                    data=df.to_dict("records"),

                    page_size=10,
                    sort_action="native",
                    filter_action="native",
                    column_selectable="single",
                    row_selectable="single",
                    selected_rows=[0],

                    style_table={
                        "overflowX": "auto",
                        "border": "2px solid #b1123f",
                        "borderRadius": "10px"
                    },

                    style_header={
                        "backgroundColor": "#b1123f",
                        "color": "white",
                        "fontWeight": "bold",
                        "textAlign": "center"
                    },

                    style_cell={
                        "textAlign": "left",
                        "minWidth": "100px",
                        "width": "150px",
                        "maxWidth": "180px",
                        "whiteSpace": "normal",
                        "fontFamily": "Arial",
                        "fontSize": "12px",
                        "padding": "8px"
                    },

                    style_data={
                        "backgroundColor": "#fffafa",
                        "color": "#333333"
                    },

                    style_data_conditional=[
                        {
                            "if": {"row_index": "odd"},
                            "backgroundColor": "#f9e6ec"
                        }
                    ]
                ),

                html.Br(),
                html.Hr(),

                html.Div(
                    style={
                        "display": "flex",
                        "gap": "20px",
                        "alignItems": "flex-start"
                    },
                    children=[

                        html.Div(
                            id="graph-id",
                            style={
                                "width": "45%",
                                "backgroundColor": "white",
                                "border": "2px solid #b1123f",
                                "borderRadius": "12px",
                                "padding": "10px",
                                "boxShadow": "0 3px 8px rgba(0,0,0,0.12)"
                            }
                        ),

                        html.Div(
                            id="map-id",
                            style={
                                "width": "55%",
                                "backgroundColor": "white",
                                "border": "2px solid #b1123f",
                                "borderRadius": "12px",
                                "padding": "10px",
                                "boxShadow": "0 3px 8px rgba(0,0,0,0.12)"
                            }
                        )
                    ]
                )
            ]
        )
    ]
)


#############################################
# Controller
#############################################

@app.callback(
    [
        Output("datatable-id", "data"),
        Output("datatable-id", "selected_rows")
    ],
    [Input("filter-type", "value")]
)
def update_dashboard(filter_type):
    """Update data table when a rescue filter is selected."""

    query = get_query(filter_type)
    records = shelter.read(query)
    filtered_df = clean_dataframe(records)

    return filtered_df.to_dict("records"), [0]


@app.callback(
    Output("datatable-id", "style_data_conditional"),
    [Input("datatable-id", "selected_columns")]
)
def update_styles(selected_columns):
    """Highlight selected columns while keeping striped row styling."""

    styles = [
        {
            "if": {"row_index": "odd"},
            "backgroundColor": "#f9e6ec"
        }
    ]

    if selected_columns is not None:
        for column in selected_columns:
            styles.append({
                "if": {"column_id": column},
                "backgroundColor": "#ffd1dc"
            })

    return styles


@app.callback(
    Output("graph-id", "children"),
    [Input("datatable-id", "derived_virtual_data")]
)
def update_graph(viewData):
    """Update pie chart based on filtered data."""

    if viewData is None or len(viewData) == 0:
        return []

    dff = pd.DataFrame.from_dict(viewData)

    breed_counts = dff["breed"].value_counts().nlargest(10).reset_index()
    breed_counts.columns = ["breed", "count"]

    fig = px.pie(
        breed_counts,
        names="breed",
        values="count",
        title="Top 10 Breed Distribution",
        color_discrete_sequence=px.colors.sequential.RdPu
    )

    fig.update_layout(
        height=450,
        paper_bgcolor="white",
        plot_bgcolor="white",
        title_font_color="#7a0026",
        title_font_size=20,
        font=dict(
            family="Arial",
            size=12,
            color="#333333"
        ),
        margin=dict(l=20, r=20, t=50, b=20)
    )

    return [
        dcc.Graph(figure=fig)
    ]


@app.callback(
    Output("map-id", "children"),
    [
        Input("datatable-id", "derived_virtual_data"),
        Input("datatable-id", "derived_virtual_selected_rows")
    ]
)
def update_map(viewData, index):
    """Update geolocation chart based on selected table row."""

    if viewData is None or len(viewData) == 0:
        return []

    dff = pd.DataFrame.from_dict(viewData)

    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]

    if row >= len(dff):
        row = 0

    return [
        dl.Map(
            style={
                "width": "100%",
                "height": "450px",
                "borderRadius": "10px"
            },
            center=[
                dff.iloc[row, 13],
                dff.iloc[row, 14]
            ],
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),

                dl.Marker(
                    position=[
                        dff.iloc[row, 13],
                        dff.iloc[row, 14]
                    ],
                    children=[
                        dl.Tooltip(str(dff.iloc[row, 4])),
                        dl.Popup([
                            html.H3("Animal Name"),
                            html.P(str(dff.iloc[row, 9]))
                        ])
                    ]
                )
            ]
        )
    ]


app.run_server(mode="jupyterlab", port=8051)